In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Find the project root
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "geo_dataset"
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

/home/utn/poli22wo/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "balanced_country_cells"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = OUTPUT_DIR / "best_model.pt"
checkpoint_path = OUTPUT_DIR / "training_checkpoint.pt"
history_path = OUTPUT_DIR / "history.csv"

In [3]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


In [4]:
countries = sorted(
    df["country"].unique()
)

country_to_index = {
    country: index
    for index, country in enumerate(countries)
}

index_to_country = {
    index: country
    for country, index in country_to_index.items()
}

df["country_index"] = df["country"].map(
    country_to_index
)

NUMBER_OF_COUNTRIES = len(countries)

print(country_to_index)

{'Belarus': 0, 'Finland': 1, 'France': 2, 'Germany': 3, 'Iceland': 4, 'Italy': 5, 'Norway': 6, 'Poland': 7, 'Spain': 8, 'Sweden': 9, 'Turkey': 10, 'United_Kingdom': 11}


Validation Split

In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


In [8]:
CELLS_PER_COUNTRY = 8

NUMBER_OF_CELLS = (
    NUMBER_OF_COUNTRIES
    * CELLS_PER_COUNTRY
)

train_df = train_df.copy()
val_df = val_df.copy()

train_df["cell_index"] = -1
val_df["cell_index"] = -1

cell_centres = np.zeros(
    (NUMBER_OF_CELLS, 2),
    dtype=np.float32,
)

cell_to_country = np.zeros(
    NUMBER_OF_CELLS,
    dtype=np.int64,
)

country_partition_trees = {}

Creating balanced geographic cells inside each country

In [9]:
def build_balanced_partition(
    coordinates,
    coordinate_indices,
    remaining_depth,
    assignments,
    next_cell,
):

    # We have reached one of the eight cells
    if remaining_depth == 0:

        cell_index = next_cell[0]
        next_cell[0] += 1

        assignments[
            coordinate_indices
        ] = cell_index

        return {
            "cell_index": cell_index,
        }

    selected_coordinates = coordinates[
        coordinate_indices
    ]

    # Approximate geographic spread in kilometres
    latitude_spread = (
        np.ptp(selected_coordinates[:, 0])
        * 111
    )

    mean_latitude = np.mean(
        selected_coordinates[:, 0]
    )

    longitude_spread = (
        np.ptp(selected_coordinates[:, 1])
        * 111
        * np.cos(np.radians(mean_latitude))
    )

    # Split along the geographically wider direction
    if latitude_spread >= longitude_spread:
        split_axis = 0
    else:
        split_axis = 1

    sorted_positions = np.argsort(
        selected_coordinates[:, split_axis],
        kind="stable",
    )

    sorted_indices = coordinate_indices[
        sorted_positions
    ]

    middle = len(sorted_indices) // 2

    left_indices = sorted_indices[:middle]
    right_indices = sorted_indices[middle:]

    left_edge = coordinates[
        left_indices[-1],
        split_axis,
    ]

    right_edge = coordinates[
        right_indices[0],
        split_axis,
    ]

    split_value = (
        left_edge + right_edge
    ) / 2

    left_tree = build_balanced_partition(
        coordinates,
        left_indices,
        remaining_depth - 1,
        assignments,
        next_cell,
    )

    right_tree = build_balanced_partition(
        coordinates,
        right_indices,
        remaining_depth - 1,
        assignments,
        next_cell,
    )

    return {
        "split_axis": split_axis,
        "split_value": split_value,
        "left": left_tree,
        "right": right_tree,
    }

In [10]:
def assign_balanced_cell(
    coordinate,
    partition_tree,
):

    current_node = partition_tree

    while "cell_index" not in current_node:

        split_axis = current_node[
            "split_axis"
        ]

        split_value = current_node[
            "split_value"
        ]

        if (
            coordinate[split_axis]
            <= split_value
        ):
            current_node = current_node[
                "left"
            ]
        else:
            current_node = current_node[
                "right"
            ]

    return current_node["cell_index"]

In [11]:
PARTITION_DEPTH = 3

for country, country_index in country_to_index.items():

    train_mask = (
        train_df["country"] == country
    )

    val_mask = (
        val_df["country"] == country
    )

    country_train_coordinates = (
        train_df.loc[
            train_mask,
            ["lat", "lng"],
        ]
        .to_numpy(dtype=np.float32)
    )

    country_val_coordinates = (
        val_df.loc[
            val_mask,
            ["lat", "lng"],
        ]
        .to_numpy(dtype=np.float32)
    )

    local_train_assignments = np.full(
        len(country_train_coordinates),
        -1,
        dtype=np.int64,
    )

    coordinate_indices = np.arange(
        len(country_train_coordinates)
    )

    next_cell = [0]

    partition_tree = build_balanced_partition(
        coordinates=country_train_coordinates,
        coordinate_indices=coordinate_indices,
        remaining_depth=PARTITION_DEPTH,
        assignments=local_train_assignments,
        next_cell=next_cell,
    )

    local_val_assignments = np.array(
        [
            assign_balanced_cell(
                coordinate,
                partition_tree,
            )
            for coordinate
            in country_val_coordinates
        ],
        dtype=np.int64,
    )

    first_cell = (
        country_index
        * CELLS_PER_COUNTRY
    )

    last_cell = (
        first_cell
        + CELLS_PER_COUNTRY
    )

    train_df.loc[
        train_mask,
        "cell_index",
    ] = (
        first_cell
        + local_train_assignments
    )

    val_df.loc[
        val_mask,
        "cell_index",
    ] = (
        first_cell
        + local_val_assignments
    )

    # Mean coordinate of each balanced cell
    for local_cell in range(
        CELLS_PER_COUNTRY
    ):

        global_cell = (
            first_cell + local_cell
        )

        cell_coordinates = (
            country_train_coordinates[
                local_train_assignments
                == local_cell
            ]
        )

        cell_centres[
            global_cell
        ] = np.mean(
            cell_coordinates,
            axis=0,
        )

    cell_to_country[
        first_cell:last_cell
    ] = country_index

    country_partition_trees[
        country
    ] = partition_tree

In [12]:
train_df["cell_index"] = (
    train_df["cell_index"].astype(int)
)

val_df["cell_index"] = (
    val_df["cell_index"].astype(int)
)

In [13]:
for country in countries:

    country_counts = (
        train_df.loc[
            train_df["country"] == country,
            "cell_index",
        ]
        .value_counts()
        .sort_index()
    )

    print(
        f"{country}: "
        f"min={country_counts.min()}, "
        f"max={country_counts.max()}, "
        f"total={country_counts.sum()}"
    )

    assert (
        country_counts.max()
        - country_counts.min()
        <= 1
    )

Belarus: min=86, max=87, total=691
Finland: min=100, max=100, total=800
France: min=100, max=100, total=800
Germany: min=100, max=100, total=800
Iceland: min=96, max=96, total=768
Italy: min=100, max=100, total=800
Norway: min=100, max=100, total=800
Poland: min=93, max=94, total=747
Spain: min=100, max=100, total=800
Sweden: min=100, max=100, total=800
Turkey: min=100, max=100, total=800
United_Kingdom: min=100, max=100, total=800


In [14]:
print("Countries:", NUMBER_OF_COUNTRIES)
print("Cells:", NUMBER_OF_CELLS)
print("Cell centres:", cell_centres.shape)
print("Cell-country mapping:", cell_to_country.shape)

Countries: 12
Cells: 96
Cell centres: (96, 2)
Cell-country mapping: (96,)


Model Verification

In [15]:
MODEL_NAME = (
    "apple/mobilevitv2-1.0-imagenet1k-256"
)

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME
)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUMBER_OF_COUNTRIES + NUMBER_OF_CELLS,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

[transformers] You passed `num_labels=108` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 36289.21it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([108])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([108, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [16]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
assert total_params <= 5_000_000

Parameters: 4,444,245


Normalizing grid cell centres

In [17]:
normalized_cell_centres = (
    cell_centres.copy()
)

normalized_cell_centres[:, 0] /= 90
normalized_cell_centres[:, 1] /= 180

cell_centres_tensor = torch.tensor(
    normalized_cell_centres,
    dtype=torch.float32,
    device=device,
)

In [18]:
cell_to_country_tensor = torch.tensor(
    cell_to_country,
    dtype=torch.long,
    device=device,
)

In [19]:
print(cell_centres_tensor.shape)
print(cell_to_country_tensor.shape)

torch.Size([96, 2])
torch.Size([96])


Image Processor and Dataset

In [20]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        country_index = torch.tensor(
            row["country_index"],
            dtype=torch.long,
        )

        cell_index = torch.tensor(
            row["cell_index"],
            dtype=torch.long,
        )

        return (
            pixel_values,
            coordinates,
            country_index,
            cell_index,
        )

In [21]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [22]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [23]:
(
    images,
    coordinates,
    country_labels,
    cell_labels,
) = next(iter(train_loader))

images = images.to(device)
coordinates = coordinates.to(device)
country_labels = country_labels.to(device)
cell_labels = cell_labels.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)
print("Country labels:", country_labels.shape)
print("Cell labels:", cell_labels.shape)

Images: torch.Size([32, 3, 256, 256])
Coordinates: torch.Size([32, 2])
Country labels: torch.Size([32])
Cell labels: torch.Size([32])


#Loss and Optimizer

In [24]:
coordinate_loss_function = nn.MSELoss()
country_loss_function = nn.CrossEntropyLoss()
cell_loss_function = nn.CrossEntropyLoss()

COUNTRY_LOSS_WEIGHT = 0.01
CELL_LOSS_WEIGHT = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

In [25]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

In [52]:
train_assigned_centres = cell_centres[
    train_df["cell_index"].to_numpy()
]

train_oracle_distances = haversine_km(
    train_df["lat"].to_numpy(),
    train_df["lng"].to_numpy(),
    train_assigned_centres[:, 0],
    train_assigned_centres[:, 1],
)

val_assigned_centres = cell_centres[
    val_df["cell_index"].to_numpy()
]

val_oracle_distances = haversine_km(
    val_df["lat"].to_numpy(),
    val_df["lng"].to_numpy(),
    val_assigned_centres[:, 0],
    val_assigned_centres[:, 1],
)

print("Training oracle metrics")
print(
    "Mean:",
    np.mean(train_oracle_distances),
)
print(
    "Median:",
    np.median(train_oracle_distances),
)

print("\nValidation oracle metrics")
print(
    "Mean:",
    np.mean(val_oracle_distances),
)
print(
    "Median:",
    np.median(val_oracle_distances),
)

Training oracle metrics
Mean: 96.78108570043307
Median: 87.84743190931168

Validation oracle metrics
Mean: 99.76049539007639
Median: 87.81281037864787


Full Train + Validation Loop

In [ ]:
start_epoch = 0
END_EPOCH = 40

history = []

best_median = float("inf")
best_epoch = 0

epochs_without_improvement = 0
patience = 4

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for (
        images,
        coordinates,
        country_labels,
        cell_labels,
    ) in training_bar:

        images = images.to(device)
        coordinates = coordinates.to(device)
        country_labels = country_labels.to(device)
        cell_labels = cell_labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=images
        ).logits

        # Split outputs
        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        # Convert logits into probabilities
        country_probabilities = torch.softmax(
            country_logits,
            dim=1,
        )

        cell_probabilities = torch.softmax(
            cell_logits,
            dim=1,
        )

        # Give every cell its country's probability
        country_weights_for_cells = country_probabilities[
            :, cell_to_country_tensor
        ]

        # Gate cells using country probabilities
        gated_cell_probabilities = (
            cell_probabilities
            * country_weights_for_cells
        )

        # Make gated probabilities sum to 1
        gated_cell_probabilities = (
            gated_cell_probabilities
            / gated_cell_probabilities.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

        # Weighted average of cell centres
        final_coordinates = (
            gated_cell_probabilities
            @ cell_centres_tensor
        )

        coordinate_loss = coordinate_loss_function(
            final_coordinates,
            coordinates,
        )

        country_loss = country_loss_function(
            country_logits,
            country_labels,
        )

        cell_loss = cell_loss_function(
            cell_logits,
            cell_labels,
        )

        loss = (
            coordinate_loss
            + COUNTRY_LOSS_WEIGHT * country_loss
            + CELL_LOSS_WEIGHT * cell_loss
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0

    all_predictions = []
    all_coordinates = []

    correct_country_predictions = 0
    correct_cell_predictions = 0
    number_of_validation_images = 0

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    with torch.no_grad():

        for (
            images,
            coordinates,
            country_labels,
            cell_labels,
        ) in validation_bar:

            images = images.to(device)
            coordinates = coordinates.to(device)
            country_labels = country_labels.to(device)
            cell_labels = cell_labels.to(device)

            outputs = model(
                pixel_values=images
            ).logits

            # Split outputs
            country_logits = outputs[
                :, :NUMBER_OF_COUNTRIES
            ]

            cell_logits = outputs[
                :, NUMBER_OF_COUNTRIES:
            ]

            # Convert logits into probabilities
            country_probabilities = torch.softmax(
                country_logits,
                dim=1,
            )

            cell_probabilities = torch.softmax(
                cell_logits,
                dim=1,
            )

            # Give every cell its country's probability
            country_weights_for_cells = (
                country_probabilities[
                    :, cell_to_country_tensor
                ]
            )

            # Gate cells using country probabilities
            gated_cell_probabilities = (
                cell_probabilities
                * country_weights_for_cells
            )

            # Make gated probabilities sum to 1
            gated_cell_probabilities = (
                gated_cell_probabilities
                / gated_cell_probabilities.sum(
                    dim=1,
                    keepdim=True,
                ).clamp_min(1e-8)
            )

            # Final coordinate prediction
            final_coordinates = (
                gated_cell_probabilities
                @ cell_centres_tensor
            )

            coordinate_loss = coordinate_loss_function(
                final_coordinates,
                coordinates,
            )

            country_loss = country_loss_function(
                country_logits,
                country_labels,
            )

            cell_loss = cell_loss_function(
                cell_logits,
                cell_labels,
            )

            loss = (
                coordinate_loss
                + COUNTRY_LOSS_WEIGHT * country_loss
                + CELL_LOSS_WEIGHT * cell_loss
            )

            total_validation_loss += loss.item()

            all_predictions.append(
                final_coordinates.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

            predicted_countries = country_logits.argmax(
                dim=1
            )

            predicted_cells = cell_logits.argmax(
                dim=1
            )

            correct_country_predictions += (
                predicted_countries == country_labels
            ).sum().item()

            correct_cell_predictions += (
                predicted_cells == cell_labels
            ).sum().item()

            number_of_validation_images += (
                cell_labels.size(0)
            )

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    country_accuracy = (
        correct_country_predictions
        / number_of_validation_images
    )

    cell_accuracy = (
        correct_cell_predictions
        / number_of_validation_images
    )

    # --------------------
    # Geographic metrics
    # --------------------
    all_predictions = np.concatenate(
        all_predictions
    )

    all_coordinates = np.concatenate(
        all_coordinates
    )

    predictions_degrees = all_predictions.copy()
    coordinates_degrees = all_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    # --------------------
    # Save history
    # --------------------
    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
        "country_accuracy": country_accuracy,
        "cell_accuracy": cell_accuracy,
    })

    # --------------------
    # Display results
    # --------------------
    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")
    print(f"Country accuracy: {country_accuracy:.2%}")
    print(f"Cell accuracy: {cell_accuracy:.2%}")

    # --------------------
    # Save best model
    # --------------------
    if median_distance < best_median:

        best_median = median_distance
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:

        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    # --------------------
    # Save resumable checkpoint
    # --------------------
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_median": best_median,
            "best_epoch": best_epoch,
            "history": history,
            "patience": patience,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
            "model_name": MODEL_NAME,
            "num_labels": (
                NUMBER_OF_COUNTRIES
                + NUMBER_OF_CELLS
            ),
            "parameter_count": total_params,
            "number_of_countries": (
                NUMBER_OF_COUNTRIES
            ),
            "cells_per_country": (
                CELLS_PER_COUNTRY
            ),
            "number_of_cells": NUMBER_OF_CELLS,
            "cell_centres": (
                cell_centres_tensor
                .detach()
                .cpu()
            ),
            "cell_to_country": (
                cell_to_country_tensor
                .detach()
                .cpu()
            ),
            "country_to_index": country_to_index,
            "index_to_country": index_to_country,
            "country_loss_weight": (
                COUNTRY_LOSS_WEIGHT
            ),
            "cell_loss_weight": (
                CELL_LOSS_WEIGHT
            ),
        },
        checkpoint_path,
    )

    # Preserve history after every epoch
    pd.DataFrame(history).to_csv(
        history_path,
        index=False,
    )

    # --------------------
    # Early stopping
    # --------------------
    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/40 - Training:   0%|          | 0/294 [00:00<?, ?it/s]

Epoch 1/40 - Validation: 100%|██████████| 74/74 [00:12<00:00,  6.08it/s]



Epoch 1 results
Training loss: 0.0702
Validation loss: 0.0600
Mean distance: 878.6 km
Median distance: 734.9 km
Within 200 km: 10.59%
Within 750 km: 50.81%
Country accuracy: 41.07%
Cell accuracy: 7.61%
Saved new best model.


Epoch 2/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.60it/s]



Epoch 2 results
Training loss: 0.0548
Validation loss: 0.0517
Mean distance: 769.0 km
Median distance: 575.4 km
Within 200 km: 16.96%
Within 750 km: 59.78%
Country accuracy: 51.79%
Cell accuracy: 15.48%
Saved new best model.


...


Epoch 25 results
Training loss: 0.0047
Validation loss: 0.0574
Mean distance: 587.4 km
Median distance: 288.5 km
Within 200 km: 40.35%
Within 750 km: 72.96%
Country accuracy: 65.52%
Cell accuracy: 32.36%
Epochs without improvement: 2


Epoch 26/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.42it/s]



Epoch 26 results
Training loss: 0.0042
Validation loss: 0.0580
Mean distance: 579.6 km
Median distance: 293.6 km
Within 200 km: 40.39%
Within 750 km: 73.72%
Country accuracy: 66.71%
Cell accuracy: 32.19%
Epochs without improvement: 3


Epoch 27/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.39it/s]



Epoch 27 results
Training loss: 0.0038
Validation loss: 0.0588
Mean distance: 573.2 km
Median distance: 290.2 km
Within 200 km: 40.26%
Within 750 km: 73.51%
Country accuracy: 66.03%
Cell accuracy: 32.82%
Epochs without improvement: 4
Early stopping.


In [27]:
best_weights = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_weights)
model.eval()

print("Loaded balanced model's best weights.")

Loaded balanced model's best weights.


In [29]:
all_country_logits = []
all_cell_logits = []
all_coordinates = []
all_country_labels = []

model.eval()

with torch.inference_mode():

    for (
        images,
        coordinates,
        country_labels,
        _,
    ) in tqdm(
        val_loader,
        desc="Collecting validation logits",
    ):

        images = images.to(device)

        outputs = model(
            pixel_values=images
        ).logits

        country_logits = outputs[
            :, :NUMBER_OF_COUNTRIES
        ]

        cell_logits = outputs[
            :, NUMBER_OF_COUNTRIES:
        ]

        all_country_logits.append(
            country_logits.cpu()
        )

        all_cell_logits.append(
            cell_logits.cpu()
        )

        all_coordinates.append(
            coordinates
        )

        all_country_labels.append(
            country_labels
        )

In [30]:
all_country_logits = torch.cat(
    all_country_logits,
    dim=0,
)

all_cell_logits = torch.cat(
    all_cell_logits,
    dim=0,
)

all_coordinates = torch.cat(
    all_coordinates,
    dim=0,
)

all_country_labels = torch.cat(
    all_country_labels,
    dim=0,
)

In [43]:
evaluation_cell_to_country = (
    cell_to_country_tensor
    .detach()
    .cpu()
)

evaluation_cell_centres = (
    cell_centres_tensor
    .detach()
    .cpu()
)

print(
    all_country_logits.device,
    evaluation_cell_to_country.device,
    evaluation_cell_centres.device,
)

cpu cpu cpu


In [45]:
def evaluate_configuration(
    name,
    country_temperature=1.0,
    cell_temperature=1.0,
    use_country_gating=True,
):

    country_probabilities = torch.softmax(
        all_country_logits
        / country_temperature,
        dim=1,
    )

    cell_probabilities = torch.softmax(
        all_cell_logits
        / cell_temperature,
        dim=1,
    )

    if use_country_gating:

        country_weights_for_cells = (
            country_probabilities[
                :, evaluation_cell_to_country
            ]
        )

        final_cell_probabilities = (
            cell_probabilities
            * country_weights_for_cells
        )

        final_cell_probabilities = (
            final_cell_probabilities
            / final_cell_probabilities.sum(
                dim=1,
                keepdim=True,
            ).clamp_min(1e-8)
        )

    else:

        final_cell_probabilities = (
            cell_probabilities
        )

    predicted_coordinates = (
        final_cell_probabilities
        @ evaluation_cell_centres
    )

    predictions_degrees = (
        predicted_coordinates.numpy().copy()
    )

    coordinates_degrees = (
        all_coordinates.numpy().copy()
    )

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    return {
        "configuration": name,
        "country_temperature": (
            country_temperature
        ),
        "cell_temperature": (
            cell_temperature
        ),
        "country_gating": (
            use_country_gating
        ),
        "mean_km": np.mean(distances),
        "median_km": np.median(distances),
        "within_200": np.mean(
            distances < 200
        ),
        "within_750": np.mean(
            distances < 750
        ),
    }

In [46]:
predicted_countries = (
    all_country_logits.argmax(dim=1)
)

country_accuracy = (
    predicted_countries
    == all_country_labels
).float().mean().item()

print(
    f"Country accuracy: "
    f"{country_accuracy:.2%}"
)

Country accuracy: 66.37%


In [47]:
temperature_values = [
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
]

temperature_results = []

In [48]:
for country_temperature in temperature_values:

    for cell_temperature in temperature_values:

        result = evaluate_configuration(
            name="Country gating",
            country_temperature=(
                country_temperature
            ),
            cell_temperature=(
                cell_temperature
            ),
            use_country_gating=True,
        )

        temperature_results.append(result)

In [49]:
for cell_temperature in temperature_values:

    result = evaluate_configuration(
        name="No country gating",
        country_temperature=1.0,
        cell_temperature=(
            cell_temperature
        ),
        use_country_gating=False,
    )

    temperature_results.append(result)

In [50]:
temperature_results_df = pd.DataFrame(
    temperature_results
)

temperature_results_df = (
    temperature_results_df
    .sort_values(
        by=[
            "median_km",
            "mean_km",
        ]
    )
    .reset_index(drop=True)
)

display(
    temperature_results_df.head(15)
)

temperature_results_df.to_csv(
    results_path,
    index=False,
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,0.75,True,581.703552,278.287933,0.410714,0.738520
1,Country gating,0.25,1.00,True,581.154358,279.882385,0.412415,0.740221
2,Country gating,0.50,0.75,True,578.884155,280.102600,0.408588,0.737670
3,Country gating,0.25,0.50,True,584.917419,281.318909,0.415816,0.733418
4,Country gating,0.25,1.25,True,581.863708,281.323029,0.407313,0.739796
5,Country gating,0.50,1.00,True,577.702881,281.456726,0.409439,0.738946
6,Country gating,0.75,0.75,True,577.324707,281.994446,0.408588,0.737245
7,Country gating,0.75,1.00,True,575.504700,282.878296,0.408588,0.738095
8,Country gating,0.50,0.50,True,583.592468,283.223633,0.413690,0.730867
9,Country gating,0.25,1.50,True,583.496338,283.574158,0.400085,0.742772


NameError: name 'results_path' is not defined

In [51]:
best_gated_result = (
    temperature_results_df[
        temperature_results_df[
            "country_gating"
        ]
    ]
    .iloc[0]
)

best_ungated_result = (
    temperature_results_df[
        ~temperature_results_df[
            "country_gating"
        ]
    ]
    .iloc[0]
)

display(
    pd.DataFrame([
        best_gated_result,
        best_ungated_result,
    ])
)

,configuration,country_temperature,cell_temperature,country_gating,mean_km,median_km,within_200,within_750
0,Country gating,0.25,0.75,True,581.703552,278.287933,0.410714,0.738520
41,No country gating,1.00,0.25,False,595.252625,298.670746,0.401361,0.718112
